### TASK 1

In [8]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set(style="whitegrid")

def bootstrap_ci(data, stat_func=np.mean, n_boot=10_000, ci_level=0.95, random_state=None):
    """
    Returns (lower, upper) percentile bootstrap confidence interval.
    
    Parameters
    ----------
    data : array-like
        Sample to resample from
    stat_func : callable
        Statistic function to compute on each resample (default: np.mean)
    n_boot : int
        Number of bootstrap resamples
    ci_level : float
        Confidence level (e.g., 0.95)
    random_state : int or None
        Seed for reproducibility
        
    Returns
    -------
    tuple: (lower, upper)
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    data = np.asarray(data)
    boot_stats = np.zeros(n_boot)
    
    for i in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_stats[i] = stat_func(sample)
    
    alpha = 1 - ci_level
    lower = np.percentile(boot_stats, 100 * alpha / 2)
    upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    
    return lower, upper 
# Validation with np.arange(1,101)
val_lower, val_upper = bootstrap_ci(np.arange(1, 101))
print(f"Validation 95% CI for 1..100: ({val_lower:.2f}, {val_upper:.2f})")

Validation 95% CI for 1..100: (44.80, 56.10)


### TASK 2


In [9]:
np.random.seed(42)
n_samples = 300

# Continuous variable: test scores ~ Normal(75, 10)
continuous_col = np.random.normal(loc=75, scale=10, size=n_samples)

# Binary variable: passed = 1/0 with probability 0.7 for 1
binary_col = np.random.binomial(n=1, p=0.7, size=n_samples)

# Convert to DataFrame for clarity
df = pd.DataFrame({
    "score": continuous_col,
    "passed": binary_col
})

# Bootstrap CIs (95%, n_boot=10000)
mean_ci = bootstrap_ci(df["score"], stat_func=np.mean, n_boot=10000, ci_level=0.95)
median_ci = bootstrap_ci(df["score"], stat_func=np.median, n_boot=10000, ci_level=0.95)
prop_ci = bootstrap_ci(df["passed"], stat_func=np.mean, n_boot=10000, ci_level=0.95)

# Summary table
summary_table = pd.DataFrame({
    "Statistic": ["Mean", "Median", "Proportion"],
    "Bootstrap Lower": [mean_ci[0], median_ci[0], prop_ci[0]],
    "Bootstrap Upper": [mean_ci[1], median_ci[1], prop_ci[1]]
})

summary_table

,Statistic,Bootstrap Lower,Bootstrap Upper
0,Mean,73.837811,76.056251
1,Median,73.549753,76.789074
2,Proportion,0.640000,0.746667


### TASK 3

In [10]:
# Sample statistics
sample_mean = df["score"].mean()
sample_std = df["score"].std(ddof=1)
n = len(df)

# Mean CI using t-distribution
mean_normal_ci = st.t.interval(0.95, df=n-1, loc=sample_mean, scale=sample_std/np.sqrt(n))

# Proportion CI using Wald formula
p_hat = df["passed"].mean()
z = 1.96  # for 95% CI
prop_se = np.sqrt(p_hat*(1-p_hat)/n)
prop_normal_ci = (p_hat - z*prop_se, p_hat + z*prop_se)

# Combine all results in comparison table
comparison_table = pd.DataFrame({
    "Statistic": ["Mean", "Proportion"],
    "Bootstrap Lower": [mean_ci[0], prop_ci[0]],
    "Bootstrap Upper": [mean_ci[1], prop_ci[1]],
    "Normal Lower": [mean_normal_ci[0], prop_normal_ci[0]],
    "Normal Upper": [mean_normal_ci[1], prop_normal_ci[1]]
})

comparison_table

,Statistic,Bootstrap Lower,Bootstrap Upper,Normal Lower,Normal Upper
0,Mean,73.837811,76.056251,73.826289,76.062740
1,Proportion,0.640000,0.746667,0.641154,0.745513


### Q: Are the two approaches giving similar intervals? Where do they diverge, if at all?
For the mean, the bootstrap CI and the t-distribution CI are very similar, because the sample size is reasonably large and the data is approximately normally distributed.
For the proportion, the bootstrap and Wald (normal-approximation) intervals are also similar in this case. However, if the sample size were smaller or the proportion closer to 0 or 1, the bootstrap would generally provide more reliable intervals.

### Q: For which statistic is the bootstrap approach especially useful, and why?
The bootstrap is particularly useful for the median (or other statistics without simple formulas) and for statistics where the underlying distribution is unknown or non-normal.
Unlike formula-based methods, bootstrap makes minimal assumptions about the distribution and can estimate confidence intervals for virtually any statistic by resampling the observed data.